<a href="https://colab.research.google.com/github/alokshah04/pgss-L4R-IL-public/blob/main/student_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imitation Learning: Teaching a Robot by Example

In this lab you will train a robot to run by **watching an expert** — no hand-coded rules, no reward engineering. The technique is called **Behavior Cloning**:

1. An expert policy runs the robot; we record what it sees and does.
2. We treat those recordings as a supervised dataset: **input = observation, label = action**.
3. We train a small neural network (MLP) to predict the expert's actions.
4. We deploy our learned policy and compare it to the expert.

The robot is **HalfCheetah** from MuJoCo — a 2D simulated cheetah that learns to run forward.

---

## ⚙️ Setup — read this before running anything

### 💻 Running locally

Do this **once** in a terminal:
```bash
conda create -n il-lab python=3.11 -y
conda activate il-lab
pip install -r requirements.txt
```
Then launch Jupyter from inside the repo folder with that environment active. **Skip Step 0.**

### ☁️ Running on Google Colab

**1. Switch to a GPU runtime:**
> Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save

**2. Run Step 0 first** and wait for `✅ Setup complete`.

> ⚠️ **Steps 1a, 1b, and 7 (video cells) do not work on Colab** — skip those. All other steps work fine.

---

## 📋 How to use this notebook

This notebook has **blanked-out sections** marked with `# TODO`. Your job is to fill them in.

**You are allowed — and encouraged — to use AI tools** (ChatGPT, Claude, Copilot, etc.) to help you write code. However, **every line of AI-generated code must have a comment written by you** explaining what that line does and why. Comments that just restate the code (e.g. `# adds x and y` next to `x + y`) do not count — your comment should demonstrate that you understand the underlying concept.

**For the architecture components** (LayerNorm and GELU), you must find a published paper that provides evidence these are the right choices for a problem like this. Include the BibTeX citation in the markdown cell above your implementation.

**Goal:** your trained policy should achieve a reward close to the expert's. The discussion questions at the end will ask you to reflect on what you observe.

---

## Step 0 — Colab Setup ☁️

> **Local users: skip this cell.**  
> **Colab users: run this cell first** and wait for `✅ Setup complete`.

In [ ]:
# ════════════════════════════════════════════════
# COLAB ONLY — local users skip this cell
# ════════════════════════════════════════════════
import subprocess, sys, os

REPO = 'pgss-L4R-IL-public'
if not os.path.isdir(REPO):
    r = subprocess.run(['git', 'clone', 'https://github.com/alokshah04/pgss-L4R-IL-public.git'],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print('Repo already cloned.')
if not os.getcwd().endswith(REPO):
    os.chdir(REPO)
print('Working directory:', os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('\n✅ Setup complete. Continue to Step 1.')

Cloning into 'pgss-L4R-IL-public'...

Working directory: /content/pgss-L4R-IL-public


---
## Step 1 — Imports and Expert 💻☁️

*(You may see a `Could not deserialize lr_schedule` warning — harmless. The policy weights load fine.)*

In [ ]:
import gymnasium as gym
from stable_baselines3 import SAC
import imageio
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import Video, display

ENV_NAME    = 'HalfCheetah-v5'
EXPERT_PATH = 'experts/HalfCheetah-v5'

expert = SAC.load(EXPERT_PATH)
print('Expert policy loaded.')

env = gym.make(ENV_NAME)
print(f'Observation space: {env.observation_space}')   # what the robot sees
print(f'Action space:      {env.action_space}')        # what the robot does
env.close()

### Step 1a — Video recorder 💻 (local only)

> **Colab users: skip Steps 1a and 1b.** Video rendering requires a display, which Colab does not have.

We use gymnasium's built-in `rgb_array` renderer to record frames.

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
def record_policy(step_fn, env_name, path, max_steps=300, seed=1, fps=30):
    """Run step_fn(obs) -> action and save an MP4 using gymnasium's rgb_array renderer."""
    env = gym.make(env_name, render_mode='rgb_array')
    obs, _ = env.reset(seed=seed)
    frames, total_reward = [], 0.0
    for _ in range(max_steps):
        action = step_fn(obs)
        obs, reward, done, truncated, _ = env.step(action)
        total_reward += reward
        frames.append(env.render())
        if done or truncated:
            break
    env.close()
    imageio.mimsave(path, frames, fps=fps)
    print(f'Saved {path}  |  reward: {total_reward:.1f}  |  frames: {len(frames)}')
    return total_reward

print('record_policy defined.')

### Step 1b — Watch the expert run 💻 (local only)

> **Colab users: skip this cell.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
def expert_step(obs):
    action, _ = expert.predict(obs, deterministic=True)
    return action

record_policy(expert_step, ENV_NAME, 'expert_demo.mp4')
display(Video('expert_demo.mp4', embed=True, width=600))

---
## Step 2 — Collect Demonstrations 💻☁️

We roll out the expert many times and store every *(observation, action)* pair it produces.

Each **trajectory** is a dictionary with three lists:
```
{
  'obs'     : [o_0, o_1, ..., o_T],   # what the robot saw at each step
  'acts'    : [a_0, a_1, ..., a_T],   # what the expert did at each step
  'rewards' : [r_0, r_1, ..., r_T],   # reward received at each step
}
```

We collect `NUM_TRAJ` trajectories in total, then split them into a **training set** and a **validation set**. The model only sees the training set during gradient updates; the validation set checks that it generalises to unseen data.

**Choose your own values for `NUM_TRAJ`, `MAX_TIMESTEPS`, and `VAL_TRAJ`** — think about how much data you need to cover the state space the robot will encounter at test time.

In [ ]:
NUM_TRAJ      = # TODO: how many total trajectories should you collect?
MAX_TIMESTEPS = # TODO: how long should each trajectory be?
VAL_TRAJ      = # TODO: how many trajectories to hold out for validation?

print(f'Collecting {NUM_TRAJ} expert trajectories...')

env = gym.make(ENV_NAME)
policy = SAC.load(EXPERT_PATH)
env.reset(seed=42)

all_trajectories = []
for ep in range(NUM_TRAJ):
    obs, _ = env.reset()
    traj = {'obs': [], 'acts': [], 'rewards': []}
    for _ in range(MAX_TIMESTEPS):
        # TODO: store the current observation in traj['obs']
        # TODO: query the expert policy for the action to take
        # TODO: store the action in traj['acts']
        # TODO: step the environment with that action
        # TODO: store the reward in traj['rewards']
        # TODO: break if the episode is done or truncated
        pass
    all_trajectories.append(traj)
env.close()

# Split into train and val
train_trajs = all_trajectories[:NUM_TRAJ - VAL_TRAJ]
val_trajs   = all_trajectories[NUM_TRAJ - VAL_TRAJ:]

print(f'Train trajectories: {len(train_trajs)}')
print(f'Val   trajectories: {len(val_trajs)}')

In [ ]:
# Inspect one trajectory — run this to check your collection loop is working
traj0 = train_trajs[0]
obs0, acts0 = np.array(traj0['obs']), np.array(traj0['acts'])
print(f'Trajectory length: {len(traj0["obs"])} steps')
print(f'Observation shape: {obs0.shape}  (T x obs_dim)')
print(f'Action shape:      {acts0.shape}  (T x act_dim)')
print(f'Total reward:      {sum(traj0["rewards"]):.1f}')
print('\nFirst observation (what the robot sees at t=0):')
print(obs0[0])
print('\nFirst expert action (what the expert does at t=0):')
print(acts0[0])

---
## Step 3 — Build a Supervised Learning Dataset 💻☁️

We flatten all trajectories into a list of *(observation, action)* pairs — exactly like any supervised learning problem:
- **Input X** = observation vector (17 numbers: joint angles & velocities)
- **Label Y** = action vector (6 numbers: one torque per joint)

PyTorch's `Dataset` class defines *what* your data is (how to index and measure it). The `DataLoader` wraps it to handle batching and shuffling automatically during training.

Choose your own `BATCH_SIZE`. Think about the tradeoff between gradient noise (small batches) and compute cost (large batches).

In [ ]:
class TrajDataset(Dataset):
    """
    Converts a list of trajectories into a PyTorch Dataset of (obs, action) pairs.
    Each sample is one timestep: the observation the robot saw and the action the expert took.
    """
    def __init__(self, trajectories):
        """
        TODO: Flatten all trajectories into two lists — one of observations, one of actions.
        Convert each list to a float32 torch.Tensor and store as self.inputs and self.labels.

        Hint: iterate over trajectories, then over each timestep within a trajectory.
        torch.tensor(np.array(list), dtype=torch.float32) converts a Python list to a tensor.
        """
        pass

    def __len__(self):
        """
        TODO: Return the total number of (obs, action) pairs in this dataset.
        """
        pass

    def __getitem__(self, idx):
        """
        TODO: Return the (observation, action) pair at index idx as a tuple.
        This is what the DataLoader calls to fetch each sample.
        """
        pass


BATCH_SIZE = # TODO: choose a batch size

train_dataset = TrajDataset(train_trajs)
val_dataset   = TrajDataset(val_trajs)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)}')
print(f'Val   samples: {len(val_dataset)}')

obs_batch, act_batch = next(iter(train_loader))
print(f'\nOne batch: observations {obs_batch.shape}, actions {act_batch.shape}')

OBS_DIM = obs_batch.shape[1]
ACT_DIM = act_batch.shape[1]
print(f'obs_dim = {OBS_DIM},  act_dim = {ACT_DIM}')

---
## Step 4 — Define the Policy Network 💻☁️

Our learned policy is a **Multi-Layer Perceptron (MLP)**. It takes an observation as input and outputs a predicted action.

The network has the following structure:

$$f(\mathbf{x}) = W_4 \cdot \sigma\!\left(\mathrm{LN}\!\left(W_3 \cdot \sigma\!\left(\mathrm{LN}\!\left(W_2 \cdot \sigma\!\left(\mathrm{LN}\!\left(W_1 \mathbf{x} + b_1\right)\right)\right) + b_2\right)\right)\right) + b_4$$

where $\mathbf{x} \in \mathbb{R}^{17}$ is the observation, $W_i, b_i$ are learned weight matrices and biases, $\mathrm{LN}(\cdot)$ is Layer Normalisation, and $\sigma$ is the GELU activation.

**Layer Normalisation** normalises the activations of each layer across the feature dimension:

$$\mathrm{LN}(\mathbf{h}) = \frac{\mathbf{h} - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \boldsymbol{\gamma} + \boldsymbol{\beta}$$

where $\mu$ and $\sigma^2$ are the mean and variance of $\mathbf{h}$, and $\boldsymbol{\gamma}, \boldsymbol{\beta}$ are learned scale and shift parameters.

**GELU** (Gaussian Error Linear Unit) is a smooth non-linearity:

$$\mathrm{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \mathrm{erf}\!\left(\frac{x}{\sqrt{2}}\right)\right]$$

where $\Phi(x)$ is the cumulative distribution function of the standard normal.

The **output layer has no activation** — actions are continuous real numbers.

---

### 📄 Citation Task

Before implementing LayerNorm and GELU, find **one paper each** that provides evidence these are good choices for a problem like this (imitation learning / behaviour cloning / continuous control). Include the BibTeX entries in the cell below, and in your implementation add a comment on the relevant line citing the paper and explaining *why* the evidence supports your choice.

*Hint for LayerNorm: look at normalisation techniques in deep RL or IL. Hint for GELU: look at the original GELU paper and at transformer/MLP literature on activation functions.*

### BibTeX Citations

**LayerNorm** — paste your citation and a one-sentence justification here:

```bibtex
% TODO: paste BibTeX for the LayerNorm paper here
```

*Justification (your own words):*

---

**GELU** — paste your citation and a one-sentence justification here:

```bibtex
% TODO: paste BibTeX for the GELU paper here
```

*Justification (your own words):*

In [ ]:
class MLP(nn.Module):
    """
    A 3-hidden-layer MLP that maps observations to actions.
    This is our learned policy: given what the robot sees, predict what to do.
    """
    def __init__(self, obs_dim, act_dim, hidden_size):
        """
        TODO: Build the network as an nn.Sequential stored in self.net.

        The architecture for each of the 3 hidden layers is:
            Linear(in, hidden_size)  ->  LayerNorm(hidden_size)  ->  GELU()

        Follow the final hidden layer with a single Linear(hidden_size, act_dim).
        No activation on the output — actions are unbounded real numbers.

        On the LayerNorm and GELU lines, add a comment that:
          (a) explains what the operation does mathematically, and
          (b) cites the paper you found above as justification.
        """
        super().__init__()
        # TODO: define self.net = nn.Sequential(...)
        pass

    def forward(self, x):
        """
        TODO: Pass x through self.net and return the result.
        """
        pass


HIDDEN_SIZE = # TODO: choose a hidden layer width
model = MLP(obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden_size=HIDDEN_SIZE)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

---
## Step 5 — Train the Policy 💻☁️

We use **supervised learning**: for each batch of observations, the model predicts actions and we penalise it for deviating from the expert.

**Loss function — Mean Squared Error:**

$$\mathcal{L}_{\text{train}} = \frac{1}{N} \sum_{i=1}^{N} \| \hat{a}_i - a_i^{\text{expert}} \|_2^2$$

where $\hat{a}_i = f_{\theta}(o_i)$ is the model's predicted action and $a_i^{\text{expert}}$ is what the expert actually did.

**Optimiser:** AdamW with a cosine learning-rate schedule that smoothly decays $\eta$ from `LR` to $0$ over `EPOCHS` steps:

$$\eta_t = \frac{\eta_0}{2}\left(1 + \cos\!\left(\frac{\pi\, t}{T}\right)\right)$$

After each epoch, compute the **validation loss** $\mathcal{L}_{\text{val}}$ on the held-out data (no gradient updates). If $\mathcal{L}_{\text{val}}$ diverges from $\mathcal{L}_{\text{train}}$, your model is overfitting.

Choose your own values for `EPOCHS` and `LR`.

**On Colab with T4 GPU:** ~2 min &nbsp;|&nbsp; **On a laptop CPU:** ~10 min

In [ ]:
EPOCHS = # TODO: how many passes over the training data?
LR     = # TODO: what initial learning rate?
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

model     = model.to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in tqdm(range(EPOCHS), desc='Training'):
    # ── training pass ────────────────────────────────────────
    model.train()
    total_train = 0.0
    for obs, acts in train_loader:
        obs, acts = obs.to(DEVICE), acts.to(DEVICE)
        # TODO: compute model predictions for this batch of observations
        # TODO: compute the MSE loss between predictions and expert actions
        # TODO: zero the gradients, run backprop, and take an optimiser step
        # TODO: accumulate total_train loss (weighted by batch size)
        pass
    scheduler.step()
    train_losses.append(total_train / len(train_dataset))

    # ── validation pass (no gradients) ───────────────────────
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for obs, acts in val_loader:
            obs, acts = obs.to(DEVICE), acts.to(DEVICE)
            # TODO: compute predictions and accumulate validation loss
            pass
    val_losses.append(total_val / len(val_dataset))

print(f'Final train loss: {train_losses[-1]:.6f}')
print(f'Final val   loss: {val_losses[-1]:.6f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses,   label='Val loss', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout(); plt.savefig('training_loss.png', dpi=120); plt.show()

---
## Step 6 — Evaluate on Test Rollouts 💻☁️

A low validation loss means the model predicts expert actions accurately on unseen *data* — but that is not the same as the robot being able to *run*. When deployed, small prediction errors **compound** over time:

$$o_t \xrightarrow{f_\theta} \hat{a}_t \xrightarrow{\text{env}} o_{t+1} \xrightarrow{f_\theta} \hat{a}_{t+1} \xrightarrow{\text{env}} \cdots$$

Each error shifts the robot into a slightly different state, which produces a slightly worse prediction, which shifts it further, and so on — this is called **compounding error** (also known as distribution shift or covariate shift).

The true test is to roll the learned policy out in the environment and measure **total reward**. We pair each learned rollout against an expert rollout from the same random seed so the comparison is fair.

In [ ]:
NUM_EVAL_TRAJ = 10
model = model.cpu().eval()

expert_returns  = []
learned_returns = []

rng = np.random.default_rng(0)
for _ in tqdm(range(NUM_EVAL_TRAJ), desc='Evaluating'):
    seed = int(rng.integers(0, 2**31))

    # ── expert rollout ───────────────────────────────────────
    env = gym.make(ENV_NAME)
    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0
    for _ in range(MAX_TIMESTEPS):
        action, _ = policy.predict(obs, deterministic=True)
        obs, reward, done, truncated, _ = env.step(action)
        ep_reward += reward
        if done or truncated: break
    env.close()
    expert_returns.append(ep_reward)

    # ── learned policy rollout ───────────────────────────────
    env = gym.make(ENV_NAME)
    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0
    with torch.no_grad():
        for _ in range(MAX_TIMESTEPS):
            # TODO: convert obs to a float32 tensor with a batch dimension
            # TODO: run the model to get a predicted action
            # TODO: convert the action back to a numpy array and step the environment
            # TODO: accumulate ep_reward and break if the episode is done
            pass
    env.close()
    learned_returns.append(ep_reward)

expert_returns  = np.array(expert_returns)
learned_returns = np.array(learned_returns)

print(f'Expert  avg return: {expert_returns.mean():.1f} +/- {expert_returns.std():.1f}')
print(f'Learned avg return: {learned_returns.mean():.1f} +/- {learned_returns.std():.1f}')
reward_gap = (expert_returns.mean() - learned_returns.mean()) / abs(expert_returns.mean())
print(f'Normalised reward gap: {reward_gap:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([expert_returns, learned_returns],
           labels=['Expert', 'Learned (ours)'], patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Total Episode Reward'); ax.set_title('Expert vs Learned Policy — Test Rollouts')
plt.tight_layout(); plt.savefig('eval_rewards.png', dpi=120); plt.show()

---
## Step 7 — Watch Your Robot Run! 💻 (local only)

> **Colab users: skip this step.**

In [ ]:
# ════════════════════════════════════════════════
# LOCAL ONLY — Colab users skip this cell
# ════════════════════════════════════════════════
model.eval()

def learned_step(obs):
    with torch.no_grad():
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        return model(obs_t).squeeze(0).numpy()

print('--- Expert ---')
record_policy(expert_step, ENV_NAME, 'expert_demo.mp4')
display(Video('expert_demo.mp4', embed=True, width=600))

print('\n--- Learned policy ---')
record_policy(learned_step, ENV_NAME, 'learned_policy.mp4')
display(Video('learned_policy.mp4', embed=True, width=600))

---
## Discussion Questions

Answer these in writing before submitting.

1. **Data matters.** How did `NUM_TRAJ` affect validation loss and final reward? At what point did adding more data stop helping, and why?

2. **Train vs val loss.** Did your two loss curves track each other? What would it mean if val loss stayed high while train loss dropped to near zero?

3. **Compounding error.** Your model can have near-zero validation loss yet still earn much lower reward than the expert. Explain precisely why, using the concept of distribution shift.

4. **Architecture choices.** What happened when you changed `HIDDEN_SIZE`? Describe the effect on train loss, val loss, and reward. What does this tell you about model capacity vs. generalisation?

5. **Epochs and overfitting.** At what point did training more stop helping? What did you observe in the val loss curve at that point?

6. **AI use reflection.** Which parts of your implementation did you use AI for? For one specific piece of AI-generated code, explain in your own words what it does and why it works — not just what the code says, but the underlying concept.

7. **Beyond imitation.** Behavior cloning only uses the expert's *actions* as supervision. What other signal could you use to improve the policy further? What would that change about the learning algorithm?

---
## A Note on Seed Variance

You may notice that running the same code twice produces noticeably different results — sometimes the robot runs well, sometimes it falls, even with identical hyperparameters. This is normal and worth understanding.

There are two independent sources of randomness in this pipeline:

**1. Training randomness** — the initial network weights are randomly initialised, and minibatches are drawn in a random order each epoch. Two training runs with different seeds can converge to different local minima, producing policies with meaningfully different deployment behaviour.

**2. Evaluation randomness** — each test episode starts from a different random initial state. A policy that averages well over many seeds may still fail on specific unlucky ones.

This means a single reward number is not a reliable measure of how good your policy is. When comparing two approaches (e.g. for Bonus 2), you should:
- Run evaluation over at least 10 episodes and report the **mean ± std**
- Consider re-running training 2–3 times with different seeds and averaging the results

If you see a large standard deviation in the box plot from Step 6, that is the compounding error problem manifesting as high variance across starting conditions — not a bug in your code.

---
## Bonus 1 — Save and Reload Your Policy 💻☁️

Once you are happy with your policy, save it so you can reload it later without retraining.

In [ ]:
torch.save(model.state_dict(), 'my_policy.pt')
print('Policy saved to my_policy.pt')

loaded_model = MLP(obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden_size=HIDDEN_SIZE)
loaded_model.load_state_dict(torch.load('my_policy.pt', map_location='cpu'))
loaded_model.eval()
print('Policy reloaded successfully.')

---
## Bonus 2 — Beat the Baseline 💻☁️

Your goal: **improve the terminal reward beyond what the default recipe achieves.**

You are free to modify anything — the architecture, the optimizer, the training pipeline, the data collection strategy, or any combination. Every change you make must be justified by a resource you found (paper, blog post, documentation, or other credible source). Include the citation and a one-sentence explanation of why you expect the change to help.

Some directions to explore (you do not have to use these — come up with your own):

- **Architecture:** deeper or wider networks, residual connections, different normalization, different activations
- **Optimizer:** different optimizer (e.g. Adam vs AdamW vs SGD with momentum), learning rate warmup, gradient clipping
- **Regularization:** dropout, weight decay strength, early stopping based on val loss
- **Data:** observation normalization, action normalization, data augmentation
- **Training objective:** L1 loss vs L2, Huber loss, weighting samples by trajectory quality

### Instructions

1. In the markdown cell below, list each change you made, the resource that justifies it, and your prediction for why it should help.
2. Re-run Steps 3–6 with your modified recipe.
3. Save your best model as `my_policy_bonus.pt`.
4. Report your best average return and compare it to the baseline."

### My Changes and Justifications

| Change | Resource (title + URL or BibTeX) | Why I expect it to help |
|--------|----------------------------------|-------------------------|
| *e.g. added residual connections* | *He et al. 2016 — Deep Residual Learning...* | *Residuals help gradients flow in deeper networks* |
| | | |
| | | |

**Baseline avg return:** (copy from Step 6)

**Bonus avg return:** (fill in after running)

In [ ]:
# ── Bonus 2: your improved recipe ────────────────────────────────────────────
# Redefine any of: MLP, optimizer, scheduler, criterion, data pipeline
# Then re-run the training loop and evaluation from Steps 3–6.
#
# Keep your original model intact above. Define your bonus model here with a
# different variable name (e.g. bonus_model) so you can compare them.

# TODO: implement your improved recipe here

# Save your best model
# torch.save(bonus_model.state_dict(), 'my_policy_bonus.pt')
# print('Bonus policy saved.')